# Ensemble: CNN + BiLSTM (ESM-2)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from contextlib import nullcontext

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Data + ESM-2

In [ ]:
seq_df = pd.read_csv('data/2018-06-06-pdb-intersect-pisces.csv', usecols=['pdb_id','seq'])
lab_df = pd.read_csv('data/2018-06-06-ss.cleaned.csv', usecols=['pdb_id','sst8','sst3'])
df = pd.merge(seq_df, lab_df, on='pdb_id', how='inner')
df['seq'] = df['seq'].str.replace('*','X')
df = df.dropna(subset=['seq','sst8','sst3']).copy()
df = df[(df['seq'].str.len()==df['sst8'].str.len()) & (df['seq'].str.len()==df['sst3'].str.len())].reset_index(drop=True)
df['len'] = df['seq'].str.len()
max_len = int(df['len'].max())

esm_model, alphabet = torch.hub.load("facebookresearch/esm:main", "esm2_t30_150M_UR50D")
esm_model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}

## 2. Embeddings

In [ ]:
sequences = [s.replace('*','X') for s in df['seq'].tolist()]
labels = df['pdb_id'].tolist() if 'pdb_id' in df.columns else list(range(len(sequences)))
data = list(zip(labels, sequences))

batch_size = 8
all_embeddings = []

for i in tqdm(range(0, len(data), batch_size), desc="Generating Embeddings"):
    batch_data = data[i:i+batch_size]
    _, _, batch_tokens = batch_converter(batch_data)
    batch_tokens = batch_tokens.to(device)
    with torch.no_grad():
        results = esm_model(batch_tokens, repr_layers=[esm_model.num_layers], return_contacts=False)
    emb = results['representations'][esm_model.num_layers][:,1:-1,:]  # strip BOS/EOS
    all_embeddings.extend([e.cpu() for e in emb])

padded_embeddings = pad_sequence(all_embeddings, batch_first=True, padding_value=0.0)
embedding_dim = padded_embeddings.shape[-1]
padded_embeddings.shape

## 3. Labels + Loaders

In [ ]:
def encode_labels(ss_labels, vocab, max_len):
    encoded = []
    for ss in ss_labels:
        ids = [vocab.get(c, -1) for c in ss]
        if len(ids) < max_len:
            ids.extend([-1] * (max_len - len(ids)))
        else:
            ids = ids[:max_len]
        encoded.append(torch.tensor(ids, dtype=torch.long))
    return pad_sequence(encoded, batch_first=True, padding_value=-1)

seq_pad_len = padded_embeddings.shape[1]
ss8_labels = encode_labels(df['sst8'], ss8_vocab, seq_pad_len)
ss3_labels = encode_labels(df['sst3'], ss3_vocab, seq_pad_len)

class ProteinDataset(Dataset):
    def __init__(self, embeddings, sst8, sst3):
        self.emb = embeddings
        self.s8 = sst8
        self.s3 = sst3
    def __len__(self): return len(self.emb)
    def __getitem__(self, idx): return self.emb[idx], self.s8[idx], self.s3[idx]

train_idx, temp_idx = train_test_split(range(len(padded_embeddings)), test_size=0.2, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_ds = ProteinDataset(padded_embeddings[train_idx], ss8_labels[train_idx], ss3_labels[train_idx])
val_ds   = ProteinDataset(padded_embeddings[val_idx],   ss8_labels[val_idx],   ss3_labels[val_idx])
test_ds  = ProteinDataset(padded_embeddings[test_idx],  ss8_labels[test_idx],  ss3_labels[test_idx])

loader_opts = dict(num_workers=2, pin_memory=True)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=False, **loader_opts)
train_loader_shuffled = DataLoader(train_ds, batch_size=16, shuffle=True, **loader_opts)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, **loader_opts)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, **loader_opts)
embedding_dim

## 4. Models

In [ ]:
class ProteinCNN(nn.Module):
    def __init__(self, input_dim=640, num_filters=128, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=num_filters, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.conv3 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters, kernel_size=7, padding=3)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout)
        self.q8_head = nn.Linear(num_filters, 8)
        self.q3_head = nn.Linear(num_filters, 3)
    def forward(self, x, mask=None):
        x = x.permute(0,2,1)
        x = self.dropout1(self.relu1(self.conv1(x)))
        x = self.dropout2(self.relu2(self.conv2(x)))
        x = self.dropout3(self.relu3(self.conv3(x)))
        x = x.permute(0,2,1)
        return self.q8_head(x), self.q3_head(x)

class ProteinBiLSTM(nn.Module):
    def __init__(self, input_dim=640, hidden_dim=256, dropout=0.3, layers=2, attn=True, attn_heads=4, attn_dropout=0.1):
        super().__init__()
        self.bilstm1 = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.bilstm2 = nn.LSTM(input_size=hidden_dim*2, hidden_size=hidden_dim, bidirectional=True, batch_first=True)
        self.use_attn = attn
        if attn:
            self.attn = nn.MultiheadAttention(embed_dim=hidden_dim*2, num_heads=attn_heads, dropout=attn_dropout, batch_first=True)
            self.ln = nn.LayerNorm(hidden_dim*2)
        self.q8_head = nn.Linear(hidden_dim*2, 8)
        self.q3_head = nn.Linear(hidden_dim*2, 3)
    def forward(self, x, mask=None):
        lengths = None
        if mask is not None:
            lengths = mask.sum(dim=1).to(torch.int64).cpu()
        if lengths is not None:
            orig_len = x.size(1)
            packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            packed_out,_ = self.bilstm1(packed)
            packed_out,_ = self.bilstm2(packed_out)
            x,_ = pad_packed_sequence(packed_out, batch_first=True, total_length=orig_len)
        else:
            x,_ = self.bilstm1(x)
            x = self.dropout(x)
            x,_ = self.bilstm2(x)
        if self.use_attn:
            key_pad = None
            if mask is not None:
                key_pad = ~mask
            attn_out,_ = self.attn(x, x, x, key_padding_mask=key_pad, need_weights=False)
            x = self.ln(x + attn_out)
        return self.q8_head(x), self.q3_head(x)

cnn = ProteinCNN(input_dim=embedding_dim).to(device)
bilstm = ProteinBiLSTM(input_dim=embedding_dim).to(device)

cnn_ckpt = 'best_cnn_model.pt'
lstm_ckpt = 'best_bilstm_esm2.pt'

def compute_accuracy(logits, labels):
    preds = logits.argmax(dim=-1)
    mask = labels >= 0
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

def train_model(model, ckpt_path):
    criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1, label_smoothing=0.05)
    criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1, label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
    clip_norm = 1.0
    best_val, epochs_no_improve, patience = -1.0, 0, 7
    epochs = 30
    for epoch in range(1, epochs+1):
        model.train(); tr_loss=tr_acc8=tr_acc3=0.0
        for emb, s8, s3 in tqdm(train_loader_shuffled, desc=f'Epoch {epoch}/{epochs}', leave=False):
            emb, s8, s3 = emb.to(device), s8.to(device), s3.to(device)
            mask = (s8 >= 0)
            optimizer.zero_grad(set_to_none=True)
            ctx = torch.amp.autocast(device_type='cuda') if torch.cuda.is_available() else nullcontext()
            with ctx:
                q8, q3 = model(emb, mask=mask)
                l8 = criterion_q8(q8.view(-1,8), s8.view(-1))
                l3 = criterion_q3(q3.view(-1,3), s3.view(-1))
                loss = l8 + 0.5*l3
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            scaler.step(optimizer); scaler.update()
            tr_loss += loss.item(); tr_acc8 += compute_accuracy(q8, s8); tr_acc3 += compute_accuracy(q3, s3)
        tr_loss /= max(1,len(train_loader_shuffled)); tr_acc8/=max(1,len(train_loader_shuffled)); tr_acc3/=max(1,len(train_loader_shuffled))
        model.eval(); va_loss=va_acc8=va_acc3=0.0
        with torch.no_grad():
            for emb, s8, s3 in val_loader:
                emb, s8, s3 = emb.to(device), s8.to(device), s3.to(device)
                mask = (s8 >= 0)
                with (torch.amp.autocast(device_type='cuda') if torch.cuda.is_available() else nullcontext()):
                    q8, q3 = model(emb, mask=mask)
                    l8 = criterion_q8(q8.view(-1,8), s8.view(-1))
                    l3 = criterion_q3(q3.view(-1,3), s3.view(-1))
                    loss = l8 + 0.5*l3
                va_loss += loss.item(); va_acc8 += compute_accuracy(q8, s8); va_acc3 += compute_accuracy(q3, s3)
        va_loss/=max(1,len(val_loader)); va_acc8/=max(1,len(val_loader)); va_acc3/=max(1,len(val_loader))
        print(f'Epoch {epoch}: Train Loss={tr_loss:.4f}, Val Loss={va_loss:.4f}')
        print(f'Train Acc Q8={tr_acc8:.4f}, Val Acc Q8={va_acc8:.4f}')
        print(f'Train Acc Q3={tr_acc3:.4f}, Val Acc Q3={va_acc3:.4f}')
        scheduler.step(va_loss)
        metric = va_acc8
        if metric > best_val:
            best_val = metric; epochs_no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print('Early stopping.')
                break
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    model.to(device); model.eval()

train_model(cnn, cnn_ckpt)
train_model(bilstm, lstm_ckpt)
'Trained'

## 5. Ensemble Tuning on Validation

In [ ]:
@torch.no_grad()
def collect_logits(loader):
    q8_c, q3_c, q8_l, q3_l, s8, s3 = [], [], [], [], [], []
    for emb, ss8, ss3 in loader:
        emb = emb.to(device)
        ss8 = ss8.to(device)
        ss3 = ss3.to(device)
        mask = (ss8 >= 0)
        q8c, q3c = cnn(emb)
        q8l, q3l = bilstm(emb, mask=mask)
        q8_c.append(q8c.cpu()); q3_c.append(q3c.cpu()); q8_l.append(q8l.cpu()); q3_l.append(q3l.cpu())
        s8.append(ss8.cpu()); s3.append(ss3.cpu())
    return (torch.cat(q8_c), torch.cat(q3_c), torch.cat(q8_l), torch.cat(q3_l), torch.cat(s8), torch.cat(s3))

def masked_accuracy(logits, labels):
    preds = logits.argmax(dim=-1)
    mask = labels >= 0
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct/total if total>0 else 0.0

val_q8_c, val_q3_c, val_q8_l, val_q3_l, val_s8, val_s3 = collect_logits(val_loader)

alphas = torch.linspace(0, 1, steps=21)
best_a8, best_acc8 = 0.5, -1
best_a3, best_acc3 = 0.5, -1
for a in alphas:
    q8_ens = a*val_q8_c + (1-a)*val_q8_l
    acc8 = masked_accuracy(q8_ens, val_s8)
    if acc8 > best_acc8:
        best_acc8, best_a8 = acc8, float(a)
    q3_ens = a*val_q3_c + (1-a)*val_q3_l
    acc3 = masked_accuracy(q3_ens, val_s3)
    if acc3 > best_acc3:
        best_acc3, best_a3 = acc3, float(a)

print(f'Best alpha Q8: {best_a8:.2f} | Val Acc: {best_acc8:.4f}')
print(f'Best alpha Q3: {best_a3:.2f} | Val Acc: {best_acc3:.4f}')

## 6. Final Evaluation on Test Set

In [ ]:
@torch.no_grad()
def evaluate(loader, a8, a3):
    total8 = total3 = 0
    correct8 = correct3 = 0
    for emb, ss8, ss3 in loader:
        emb = emb.to(device); ss8 = ss8.to(device); ss3 = ss3.to(device)
        mask = (ss8 >= 0)
        q8c, q3c = cnn(emb)
        q8l, q3l = bilstm(emb, mask=mask)
        q8 = a8*q8c + (1-a8)*q8l
        q3 = a3*q3c + (1-a3)*q3l
        p8 = q8.argmax(dim=-1); p3 = q3.argmax(dim=-1)
        m8 = ss8 >= 0; m3 = ss3 >= 0
        correct8 += (p8[m8] == ss8[m8]).sum().item(); total8 += m8.sum().item()
        correct3 += (p3[m3] == ss3[m3]).sum().item(); total3 += m3.sum().item()
    return (correct8/max(1,total8), correct3/max(1,total3))

test_acc8, test_acc3 = evaluate(test_loader, best_a8, best_a3)
print(f'Test Accuracy Q8: {test_acc8:.4f}')
print(f'Test Accuracy Q3: {test_acc3:.4f}')
